In [ ]:
!pip install wandb

In [ ]:
import wandb

wandb.login()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import os
from PIL import Image
import numpy as np
import copy
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple
import torchvision.transforms as transforms

In [ ]:
# ============ CONFIGURATION CLASS ============
@dataclass
class ESRCNNConfig:
    """Training configuration for Enhanced SRCNN"""
    # Model
    model_name: str = 'esrcnn_face'
    scale_factor: int = 2
    num_features: int = 64
    num_residual_blocks: int = 10
    use_global_skip: bool = True
    input_channels: int = 3
    crop_size: int = 48
    
    # Training
    batch_size: int = 16
    num_epochs: int = 150
    learning_rate: float = 1e-4
    lr_scheduler: str = 'step'
    lr_decay_epochs: List[int] = field(default_factory=lambda: [50, 100, 130])
    lr_decay_factor: float = 0.5
    optimizer: str = 'adam'
    weight_decay: float = 1e-4
    betas: tuple = (0.9, 0.999)
    
    # Loss
    loss_pixel_weight: float = 1.0
    loss_perceptual_weight: float = 0.1
    loss_ssim_weight: float = 0.0
    use_perceptual_loss: bool = True
    pixel_loss_type: str = 'l1'
    perceptual_layers: List[int] = field(default_factory=lambda: [3, 8, 17, 26])
    
    # Hardware & Checkpointing
    data_dir: str = 'dataset/ffhq'
    num_workers: int = 4
    val_interval: int = 5
    save_dir: str = 'checkpoints/esrcnn'
    save_interval: int = 10
    log_interval: int = 10
    device: str = 'cuda'
    seed: int = 42
    mixed_precision: bool = True
    
    def to_dict(self) -> dict:
        return self.__dict__


# ============ MODEL COMPONENTS ============
class ResidualBlock(nn.Module):
    """Residual Block with Batch Normalization"""
    def __init__(self, channels: int, kernel_size: int = 3):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm2d(channels)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + identity
        out = self.relu(out)
        return out


class UpsampleBlock(nn.Module):
    """Efficient Sub-Pixel Convolution Upsampling"""
    def __init__(self, in_channels: int, scale_factor: int):
        super(UpsampleBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * (scale_factor ** 2), kernel_size=3, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(scale_factor)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.pixel_shuffle(x)
        x = self.relu(x)
        return x


class EnhancedSRCNN(nn.Module):
    """Enhanced SRCNN for Face Super-Resolution"""
    def __init__(
        self, 
        in_channels: int = 3,
        num_features: int = 64,
        num_residual_blocks: int = 10,
        scale_factor: int = 2,
        use_global_skip: bool = True
    ):
        super(EnhancedSRCNN, self).__init__()
        
        self.scale_factor = scale_factor
        self.use_global_skip = use_global_skip
        
        # Feature extraction
        self.feature_extraction = nn.Sequential(
            nn.Conv2d(in_channels, num_features, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
        # Residual blocks
        residual_layers = []
        for _ in range(num_residual_blocks):
            residual_layers.append(ResidualBlock(num_features, kernel_size=3))
        self.residual_blocks = nn.Sequential(*residual_layers)
        
        # Bottleneck
        self.bottleneck = nn.Sequential(
            nn.Conv2d(num_features, num_features, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features)
        )
        
        # Upsampling
        upsampling_layers = []
        if scale_factor in [2, 4, 8]:
            num_upsample_blocks = int(np.log2(scale_factor))
            for _ in range(num_upsample_blocks):
                upsampling_layers.append(UpsampleBlock(num_features, scale_factor=2))
        else:
            raise ValueError(f"Scale factor {scale_factor} not supported. Use 2, 4, or 8.")
        
        self.upsampling = nn.Sequential(*upsampling_layers)
        
        # Reconstruction
        self.reconstruction = nn.Conv2d(num_features, in_channels, kernel_size=3, padding=1)
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Kaiming initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_global_skip:
            x_upsampled = F.interpolate(x, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)
        
        features = self.feature_extraction(x)
        residual_output = self.residual_blocks(features)
        bottleneck_output = self.bottleneck(residual_output)
        bottleneck_output = bottleneck_output + features
        upsampled = self.upsampling(bottleneck_output)
        output = self.reconstruction(upsampled)
        
        if self.use_global_skip:
            output = output + x_upsampled
        
        return output


# ============ LOSS FUNCTIONS ============
class CharbonnierLoss(nn.Module):
    """Charbonnier Loss (smoother L1)"""
    def __init__(self, epsilon: float = 1e-3):
        super().__init__()
        self.epsilon = epsilon
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        diff = pred - target
        loss = torch.sqrt(diff ** 2 + self.epsilon ** 2)
        return loss.mean()


class SSIMLoss(nn.Module):
    """Structural Similarity Index Loss"""
    def __init__(self, window_size: int = 11, sigma: float = 1.5):
        super().__init__()
        self.window_size = window_size
        self.sigma = sigma
        
        kernel = torch.arange(window_size, dtype=torch.float32) - (window_size - 1) / 2
        kernel = torch.exp(-kernel.pow(2.0) / (2 * sigma ** 2))
        kernel = kernel / kernel.sum()
        kernel_2d = kernel.view(-1, 1) * kernel.view(1, -1)
        self.register_buffer('kernel', kernel_2d.unsqueeze(0).unsqueeze(0))
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0) -> torch.Tensor:
        B, C, H, W = pred.shape
        kernel = self.kernel.repeat(C, 1, 1, 1).to(pred.device)
        
        mu1 = F.conv2d(pred, kernel, padding=self.window_size // 2, groups=C)
        mu2 = F.conv2d(target, kernel, padding=self.window_size // 2, groups=C)
        
        mu1_sq = mu1 ** 2
        mu2_sq = mu2 ** 2
        mu1_mu2 = mu1 * mu2
        
        sigma1_sq = F.conv2d(pred ** 2, kernel, padding=self.window_size // 2, groups=C) - mu1_sq
        sigma2_sq = F.conv2d(target ** 2, kernel, padding=self.window_size // 2, groups=C) - mu2_sq
        sigma12 = F.conv2d(pred * target, kernel, padding=self.window_size // 2, groups=C) - mu1_mu2
        
        c1 = (0.01 * data_range) ** 2
        c2 = (0.03 * data_range) ** 2
        
        ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / \
                   ((mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2))
        
        loss = 1 - ssim_map.mean()
        return loss


class PerceptualLoss(nn.Module):
    """Perceptual Loss using VGG19 features"""
    def __init__(self, feature_layers: list = [3, 8, 17, 26], use_input_norm: bool = True):
        super(PerceptualLoss, self).__init__()
        
        try:
            from torchvision.models import vgg19, VGG19_Weights
            vgg = vgg19(weights=VGG19_Weights.IMAGENET1K_V1)
        except:
            from torchvision.models import vgg19
            vgg = vgg19(pretrained=True)
        
        self.feature_layers = feature_layers
        self.use_input_norm = use_input_norm
        
        self.feature_extractors = nn.ModuleList()
        current_layer = 0
        for layer_idx in feature_layers:
            self.feature_extractors.append(
                nn.Sequential(*list(vgg.features.children())[current_layer:layer_idx+1])
            )
            current_layer = layer_idx + 1
        
        for param in self.parameters():
            param.requires_grad = False
        
        if use_input_norm:
            self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
            self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    
    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        if self.use_input_norm:
            pred = (pred - self.mean) / self.std
            target = (target - self.mean) / self.std
        
        loss = 0.0
        pred_features = pred
        target_features = target
        
        for extractor in self.feature_extractors:
            pred_features = extractor(pred_features)
            target_features = extractor(target_features)
            loss += F.l1_loss(pred_features, target_features)
        
        return loss / len(self.feature_extractors)

In [ ]:
# ESRCNN Training configuration
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

# Data paths
csv_path = "/kaggle/input/celeba-dataset/list_eval_partition.csv"
img_dir = "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba"

# Model configuration
scale_factor = 2
num_features = 64
num_residual_blocks = 10

# Training hyperparameters
learning_rate = 1e-4
batch_size = 16
num_epochs = 150

# Loss configuration
use_perceptual_loss = True
loss_perceptual_weight = 0.1
loss_pixel_weight = 1.0
pixel_loss_type = 'l1'  # 'l1' or 'mse'

# Hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mixed_precision = True
seed = 42

# Logging and saving
save_dir = 'checkpoints/esrcnn'
val_interval = 5
save_interval = 10
log_interval = 10

# Wandb configuration
wandb.init(project="ESRCNN-Face-SR", config={
    "learning_rate": learning_rate,
    "batch_size": batch_size,
    "num_epochs": num_epochs,
    "scale_factor": scale_factor,
    "num_features": num_features,
    "num_residual_blocks": num_residual_blocks,
    "use_perceptual_loss": use_perceptual_loss,
    "loss_perceptual_weight": loss_perceptual_weight,
}, reinit="finish_previous")

In [ ]:
# ============ DATASET & DATALOADER SETUP ============

# Read CSV with splits
dataframe = pd.read_csv(csv_path)

# Each dataframe contains the image file names belonging to that split
train_df = dataframe[dataframe['partition'] == 0]
validation_df = dataframe[dataframe['partition'] == 1]
test_df = dataframe[dataframe['partition'] == 2]

# Function that transforms from Pillow RGB to PyTorch tensor and normalizes to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor()
])

class CelebADataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, img_dir: str, transform_func=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform_func = transform_func

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_filename = self.dataframe.loc[idx, 'image_id']
        img_path = os.path.join(self.img_dir, img_filename)

        hr_image = Image.open(img_path).convert('RGB')

        hr_adjusted_width = (hr_image.width // scale_factor) * scale_factor
        hr_adjusted_height = (hr_image.height // scale_factor) * scale_factor
        hr_image = hr_image.resize((hr_adjusted_width, hr_adjusted_height), resample=Image.BICUBIC)

        # create LR from HR
        lr_image = hr_image.resize((hr_adjusted_width // scale_factor, hr_adjusted_height // scale_factor),
                                resample=Image.BICUBIC)
        
        # Use PyTorch transforms to convert to tensor and normalize
        hr_image = transform(hr_image)
        lr_image = transform(lr_image)

        return lr_image, hr_image

In [ ]:
# ============ CREATE DATA LOADERS ============

# Create datasets and dataloaders
train_dataset = CelebADataset(train_df, img_dir, transform)
val_dataset = CelebADataset(validation_df, img_dir, transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=4)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# ============ INITIALIZE MODEL & TRAINER ============

# Create config
config = ESRCNNConfig(
    scale_factor=scale_factor,
    num_residual_blocks=num_residual_blocks,
    num_features=num_features,
    batch_size=batch_size,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    use_perceptual_loss=use_perceptual_loss,
    loss_perceptual_weight=loss_perceptual_weight,
    loss_pixel_weight=loss_pixel_weight,
    pixel_loss_type=pixel_loss_type,
    device=device.type,
    mixed_precision=mixed_precision,
    save_dir=save_dir,
    val_interval=val_interval,
    save_interval=save_interval,
    log_interval=log_interval,
    seed=seed,
)

# Set seeds
torch.manual_seed(config.seed)
np.random.seed(config.seed)

# Create model
model = EnhancedSRCNN(
    in_channels=config.input_channels,
    num_features=config.num_features,
    num_residual_blocks=config.num_residual_blocks,
    scale_factor=config.scale_factor,
    use_global_skip=True
).to(device)

# Setup optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=config.learning_rate,
    betas=config.betas,
    weight_decay=config.weight_decay
)

# Setup learning rate scheduler
scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=config.lr_decay_epochs,
    gamma=config.lr_decay_factor
)

# Setup losses
pixel_loss_fn = CharbonnierLoss() if config.pixel_loss_type == 'l1' else nn.MSELoss()
perceptual_loss_fn = PerceptualLoss(feature_layers=config.perceptual_layers).to(device) if config.use_perceptual_loss else None
ssim_loss_fn = SSIMLoss() if config.loss_ssim_weight > 0 else None

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler() if (config.mixed_precision and device.type == 'cuda') else None

# Checkpoint directory
Path(config.save_dir).mkdir(parents=True, exist_ok=True)

print(f"\nModel Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")
print(f"Mixed Precision: {scaler is not None}\n")


# ============ TRAINING LOOP ============

def compute_loss(pred, target):
    """Compute combined loss"""
    losses = {}
    
    # Pixel loss
    pixel_loss = pixel_loss_fn(pred, target)
    losses['pixel'] = pixel_loss * config.loss_pixel_weight
    
    # Perceptual loss
    if perceptual_loss_fn is not None:
        percep_loss = perceptual_loss_fn(pred, target)
        losses['perceptual'] = percep_loss * config.loss_perceptual_weight
    
    # SSIM loss
    if ssim_loss_fn is not None:
        ssim_loss = ssim_loss_fn(pred, target)
        losses['ssim'] = ssim_loss * config.loss_ssim_weight
    
    total_loss = sum(losses.values())
    losses['total'] = total_loss
    
    return losses


def train_epoch(epoch_num, train_loader):
    """Train for one epoch"""
    model.train()
    
    total_loss = 0.0
    num_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch_num+1}")
    
    for batch_idx, (lr_images, hr_images) in enumerate(pbar):
        lr_images = lr_images.to(device)
        hr_images = hr_images.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass with mixed precision
        if scaler is not None:
            with torch.cuda.amp.autocast():
                pred = model(lr_images)
                losses = compute_loss(pred, hr_images)
                loss = losses['total']
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(lr_images)
            losses = compute_loss(pred, hr_images)
            loss = losses['total']
            
            loss.backward()
            optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if batch_idx % config.log_interval == 0:
            loss_str = f"Loss: {loss.item():.6f}"
            if 'perceptual' in losses:
                loss_str += f" | Percep: {losses['perceptual'].item():.6f}"
            pbar.set_postfix_str(loss_str)
    
    return total_loss / max(num_batches, 1)


@torch.no_grad()
def validate(val_loader):
    """Validate model"""
    model.eval()
    
    total_loss = 0.0
    total_psnr = 0.0
    num_batches = 0
    
    for lr_images, hr_images in tqdm(val_loader, desc="Validation"):
        lr_images = lr_images.to(device)
        hr_images = hr_images.to(device)
        
        pred = model(lr_images)
        losses = compute_loss(pred, hr_images)
        
        # Compute PSNR
        mse = F.mse_loss(pred, hr_images)
        psnr = 10 * torch.log10(1.0 / (mse + 1e-8))
        
        total_loss += losses['total'].item()
        total_psnr += psnr.item()
        num_batches += 1
    
    return {
        'val_loss': total_loss / max(num_batches, 1),
        'val_psnr': total_psnr / max(num_batches, 1),
    }


def save_checkpoint(epoch_num, best_val_loss, is_best=False):
    """Save checkpoint"""
    checkpoint = {
        'epoch': epoch_num,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_loss': best_val_loss,
    }
    
    if is_best:
        checkpoint_path = Path(config.save_dir) / 'best_model.pth'
    else:
        checkpoint_path = Path(config.save_dir) / f'checkpoint_epoch_{epoch_num:04d}.pth'
    
    torch.save(checkpoint, checkpoint_path)
    print(f"  💾 Saved to {checkpoint_path}")


# Main training loop
best_val_loss = float('inf')
best_epoch = 0

print(f"\n{'='*70}")
print(f"Starting Enhanced SRCNN Training")
print(f"{'='*70}\n")

for epoch in range(num_epochs):
    # Train
    train_loss = train_epoch(epoch, train_loader)
    
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.6f}")
    print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.2e}")
    
    # Log to wandb
    wandb.log({
        "train_loss": train_loss,
        "epoch": epoch + 1,
        "lr": optimizer.param_groups[0]['lr'],
    })
    
    # Validate
    if (epoch + 1) % val_interval == 0:
        val_metrics = validate(val_loader)
        
        print(f"  Val Loss: {val_metrics['val_loss']:.6f}")
        print(f"  Val PSNR: {val_metrics['val_psnr']:.2f} dB")
        
        wandb.log({
            "val_loss": val_metrics['val_loss'],
            "val_psnr": val_metrics['val_psnr'],
            "epoch": epoch + 1
        })
        
        # Save best model
        if val_metrics['val_loss'] < best_val_loss:
            best_val_loss = val_metrics['val_loss']
            best_epoch = epoch + 1
            save_checkpoint(epoch, best_val_loss, is_best=True)
            print(f"  ✅ New best model found!")
    
    # Learning rate scheduling
    scheduler.step()
    
    # Save periodic checkpoint
    if (epoch + 1) % save_interval == 0:
        save_checkpoint(epoch, best_val_loss, is_best=False)

print(f"\n{'='*70}")
print(f"Training completed!")
print(f"Best validation loss: {best_val_loss:.6f} (Epoch {best_epoch})")
print(f"Checkpoints saved to: {config.save_dir}")
print(f"{'='*70}\n")

# Finish wandb
wandb.finish()